# 03 - Report
Renders `report.md` from `findings.json` (same `RUN_ID` folder). The branded PDF stage needs Node.js + Puppeteer and is skipped in Fabric; generate it later on a workstation with `python reports/_generate_pdf.py`.

## Parameters

In [ ]:
# Parameters (injected by the pipeline)
GITHUB_REPO_URL = "https://github.com/biro98/fabric-architecture-review.git"
GITHUB_BRANCH = "main"
GITHUB_REF = ""  # pinned release ref from the pipeline (blank = GITHUB_BRANCH tip)
CLIENT_NAME = "Contoso"
ENGAGEMENT_NAME = "Fabric Architecture Review"
REVIEWER_NAME = ""
RUN_ID = "latest"

## Clone framework + install dependencies

In [ ]:
import os, sys, shutil, subprocess
WORK_ROOT = "/tmp/fabric-arch-review-run"
REPO_DIR = os.path.join(WORK_ROOT, "repo")
os.makedirs(WORK_ROOT, exist_ok=True)
_url = GITHUB_REPO_URL
_ref = (GITHUB_REF or "").strip() or GITHUB_BRANCH
def _git(args, **kw):
    r = subprocess.run(["git"] + args, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        out = ((r.stderr or "") + (r.stdout or "")).strip()
        raise RuntimeError("git " + " ".join(args) + " failed (rc=" + str(r.returncode) + "): " + out)
    return r

subprocess.run(["git", "config", "--global", "--add", "safe.directory", REPO_DIR])

def _fresh_clone():
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    _git(["clone", "--branch", _ref, "--depth", "1", _url, REPO_DIR])

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    try:
        _git(["-C", REPO_DIR, "remote", "set-url", "origin", _url])
        _git(["-C", REPO_DIR, "fetch", "--depth", "1", "origin", _ref])
        _git(["-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"])
    except RuntimeError:
        _fresh_clone()
else:
    _fresh_clone()
del _url
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
_pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")])
if _pip.returncode != 0:
    print("WARNING: pip install exited", _pip.returncode, "- a benign dependency-resolver warning is expected on Fabric; continuing.")
print("Repo + deps ready at", REPO_DIR)

## Render the Markdown report

In [ ]:
import os, subprocess, sys
os.environ["CLIENT_NAME"] = CLIENT_NAME
os.environ["ENGAGEMENT_NAME"] = ENGAGEMENT_NAME
os.environ["REVIEWER_NAME"] = REVIEWER_NAME
LH = "/lakehouse/default/Files"
_BASE = os.path.join(LH, "fabric-arch-review")
if RUN_ID == "latest" and os.path.isdir(_BASE):
    _runs = [d for d in os.listdir(_BASE) if os.path.isdir(os.path.join(_BASE, d, "raw"))]
    if _runs:
        RUN_ID = max(_runs, key=lambda d: os.path.getmtime(os.path.join(_BASE, d)))
        print("Resolved RUN_ID=latest ->", RUN_ID)
OUT_DIR = os.path.join(LH, "fabric-arch-review", RUN_ID)
RAW_DIR = os.path.join(OUT_DIR, "raw")
findings = os.path.join(OUT_DIR, "findings.json")
if not os.path.isfile(findings):
    raise RuntimeError("findings.json not found -- run the Analyze stage first.")
report_md = os.path.join(OUT_DIR, "report.md")
r = subprocess.run([sys.executable, "-m", "reports.render_report",
                    "--findings", findings, "--out", report_md, "--raw-dir", RAW_DIR], cwd=REPO_DIR)
print("render_report exit:", r.returncode)
if os.path.isfile(report_md):
    print("\n--- report.md (preview) ---\n")
    print(open(report_md, encoding="utf-8").read()[:3000])
try:
    import notebookutils
    notebookutils.notebook.exit("report:" + report_md)
except Exception:
    pass